# 01 — Data exploration

Which `nba_api` endpoints give us clean, league-wide, time-correct player-season data? Everything here is tested on **one season** before any bulk download.

In [1]:
import pandas as pd
import numpy as np

## Step 1B — League-wide player-season data: `LeagueDashPlayerStats`

One call returns **every player for one season**, so 15 seasons × a few measure types is about 50–100 calls total.

Design choice: request `Totals`, not `Per36`. Totals add up across seasons and stints, so we can derive per-36, per-100, rolling multi-season rates and shrinkage estimates ourselves from one consistent source. Advanced rates (USG%, AST%, …) need team and opponent possession data we don't have, so we take those as given.

⚠️ With `Advanced`, `MIN` is **per game** even when `Totals` is requested. We take minutes from `Base` only.

In [2]:
import time
from nba_api.stats.endpoints import leaguedashplayerstats as ldps

SEASON = "2018-19"

def fetch_league_players(season, measure, per_mode="Totals"):
    """One call = every player in the league for one season (consolidated across teams)."""
    df = ldps.LeagueDashPlayerStats(
        season=season,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense=measure,
        per_mode_detailed=per_mode,
        timeout=60,
    ).get_data_frames()[0]
    time.sleep(1.0)  # be polite to stats.nba.com; it rate-limits aggressively
    return df

base = fetch_league_players(SEASON, "Base")
adv = fetch_league_players(SEASON, "Advanced")

print(base.shape, adv.shape)
print([c for c in base.columns if not c.endswith("_RANK")])

(530, 67) (530, 79)
['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'TEAM_COUNT']


In [3]:
adv_cols = ["PLAYER_ID", "POSS", "PACE", "USG_PCT", "TS_PCT", "EFG_PCT",
            "AST_PCT", "AST_RATIO", "TM_TOV_PCT", "OREB_PCT", "DREB_PCT", "REB_PCT",
            "OFF_RATING", "DEF_RATING", "NET_RATING", "PIE"]

df = base.merge(adv[adv_cols], on="PLAYER_ID", how="outer",
                validate="one_to_one", indicator=True)

print("shape:", df.shape)
print("join:", df["_merge"].value_counts().to_dict())
print("teams per player:", df["TEAM_COUNT"].value_counts().sort_index().to_dict())
print("players with >= N minutes:",
      {t: int((df["MIN"] >= t).sum()) for t in [0, 100, 250, 500, 1000, 1500]})

shape: (530, 83)
join: {'both': 530, 'left_only': 0, 'right_only': 0}
teams per player: {1: 444, 2: 80, 3: 6}
players with >= N minutes: {0: 530, 100: 461, 250: 413, 500: 361, 1000: 276, 1500: 183}


In [4]:
cols = ["PLAYER_NAME", "GP", "MIN", "FGA", "FG3A", "FG3_PCT", "TS_PCT", "USG_PCT"]
display(df.sort_values("TS_PCT", ascending=False)[cols].head(5))
display(df[df["PLAYER_NAME"] == "Shai Gilgeous-Alexander"][cols + ["AST_PCT", "POSS"]])

,PLAYER_NAME,GP,MIN,FGA,FG3A,FG3_PCT,TS_PCT,USG_PCT
306,Jordan Sibert,1,4.233333,1,1,1.0,1.500,0.083
328,Zhou Qi,1,0.955000,1,0,0.0,1.000,0.333
456,Trevon Duval,3,5.603333,3,1,1.0,0.833,0.231
179,Scott Machado,4,19.166667,6,1,1.0,0.776,0.130
321,Damian Jones,24,409.618333,74,0,0.0,0.720,0.113


,PLAYER_NAME,GP,MIN,FGA,FG3A,FG3_PCT,TS_PCT,USG_PCT,AST_PCT,POSS
459,Shai Gilgeous-Alexander,82,2173.906667,716,139,0.367,0.554,0.182,0.179,4669


### Step 1B findings

- **One row per player-season**: traded players are already consolidated (`TEAM_COUNT` > 1), and `TEAM_ABBREVIATION` is their *last* team. That's fine for player features, wrong for team context.
- **`_PCT` columns are fractions** (USG 0.182, not 18.2).
- **Undefined percentages come back as `0.0`, not NaN**: Zhou Qi and Damian Jones show `FG3_PCT = 0.0` on 0 attempts. When cleaning, we'll recompute every percentage from makes and attempts, so 0/0 becomes NaN.
- **Small samples dominate the top of every rate stat** (e.g. TS% 1.500 on 1 FGA). Plan: empirical-Bayes shrinkage, with each stat shrunk according to its own denominator (FGA for TS%, possessions for USG%), rather than a hard minutes cutoff.
- **Missing from this endpoint:** height, weight, position, birth date, draft info. Those come in Step 1C.

## Step 1C — Bio, draft, and the remaining stat families

Questions:
1. Is height/weight **time-correct** (what was listed *that* season), or today's value copied backwards? The second would be leakage.
2. Where do position, draft position and physical measurements come from?
3. Can we get STL%, BLK%, TOV% and shot-profile features without leaving NBA.com?

In [5]:
from nba_api.stats.endpoints import leaguedashplayerbiostats

def fetch_bio(season):
    df = leaguedashplayerbiostats.LeagueDashPlayerBioStats(
        season=season, per_mode_simple="Totals", timeout=60
    ).get_data_frames()[0]
    time.sleep(1.0)
    return df

bio = {s: fetch_bio(s) for s in ["2018-19", "2019-20"]}
print(bio["2018-19"].shape, bio["2018-19"]["PLAYER_ID"].is_unique)
print("same players as Base:", set(bio["2018-19"]["PLAYER_ID"]) == set(base["PLAYER_ID"]))
bio["2018-19"][["PLAYER_NAME", "AGE", "PLAYER_HEIGHT_INCHES", "PLAYER_WEIGHT",
                "COUNTRY", "DRAFT_YEAR", "DRAFT_ROUND", "DRAFT_NUMBER"]].head()

(530, 23) True
same players as Base: True


,PLAYER_NAME,AGE,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,COUNTRY,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER
0,Aaron Gordon,23.0,81,220,USA,2014,1,4
1,Aaron Holiday,22.0,73,185,USA,2018,1,23
2,Abdel Nader,25.0,78,225,Egypt,2016,2,58
3,Al Horford,33.0,82,245,Dominican Republic,2007,1,3
4,Al-Farouq Aminu,28.0,81,220,USA,2010,1,8


### The 2019 height-measurement break

From 2019-20 the NBA required officially measured, barefoot heights. If listed height is time-correct, we should see a one-off drop between these two seasons and stability elsewhere (I checked 2016-17 → 2018-19 and 2019-20 → 2020-21 separately: about 95% unchanged in each).

In [6]:
h = (pd.concat([df.assign(SEASON=s) for s, df in bio.items()])
       .pivot_table(index="PLAYER_ID", columns="SEASON", values="PLAYER_HEIGHT_INCHES"))
diff = (h["2019-20"] - h["2018-19"]).dropna()
print(f"n={len(diff)}  shorter={(diff < 0).mean():.0%}  same={(diff == 0).mean():.0%}  taller={(diff > 0).mean():.0%}")
print("mean change (in):", round(diff.mean(), 2))
diff.value_counts().sort_index()

n=400  shorter=56%  same=39%  taller=5%
mean change (in): -0.57


-2.0     32
-1.0    190
 0.0    157
 1.0     19
 2.0      1
 3.0      1
Name: count, dtype: int64

### `PlayerIndex` is NOT time-correct

It accepts a `season` parameter, but returns today's bio values. Jokić's 2018-19 bio row above says 7-0 / 250; here he is 6-11 / 284 (his current listing). LaMelo Ball appears in a 2018-19 query even though he entered the league in 2020. It is also the only NBA.com source of `POSITION`, which is therefore current-only.

In [7]:
from nba_api.stats.endpoints import playerindex

pi = playerindex.PlayerIndex(season="2018-19", historical_nullable="1", timeout=60).get_data_frames()[0]
time.sleep(1.0)
ids = {203999: "Jokic", 1630163: "LaMelo", 1628983: "SGA"}
display(pi[pi["PERSON_ID"].isin(ids)][["PLAYER_FIRST_NAME", "PLAYER_LAST_NAME", "POSITION", "HEIGHT", "WEIGHT", "DRAFT_YEAR"]])
display(bio["2018-19"][bio["2018-19"]["PLAYER_ID"].isin(ids)][["PLAYER_NAME", "PLAYER_HEIGHT", "PLAYER_WEIGHT"]])

,PLAYER_FIRST_NAME,PLAYER_LAST_NAME,POSITION,HEIGHT,WEIGHT,DRAFT_YEAR
212,LaMelo,Ball,G,6-7,180,2020.0
1638,Shai,Gilgeous-Alexander,G,6-6,195,2018.0
2385,Nikola,Jokić,C,6-11,284,2014.0


,PLAYER_NAME,PLAYER_HEIGHT,PLAYER_WEIGHT
388,Nikola Jokić,7-0,250
444,Shai Gilgeous-Alexander,6-6,181


### Draft history and combine measurements

`DraftHistory` returns every pick since 1947 in one call. `DraftCombineStats` returns one call per draft year: pre-draft, so time-correct by construction, but it only covers players who attended the combine.

In [8]:
from nba_api.stats.endpoints import drafthistory, draftcombinestats

draft = drafthistory.DraftHistory(timeout=60).get_data_frames()[0]
time.sleep(1.0)
print("DraftHistory:", draft.shape, draft["SEASON"].min(), "→", draft["SEASON"].max())

combine = {}
for s in ["2018-19", "2025-26"]:
    combine[s] = draftcombinestats.DraftCombineStats(season_all_time=s, timeout=60).get_data_frames()[0]
    time.sleep(1.0)
    print("Combine", s, combine[s].shape)

combine_cols = ["PLAYER_ID", "PLAYER_NAME", "POSITION", "HEIGHT_WO_SHOES", "WEIGHT",
                "WINGSPAN", "STANDING_REACH", "MAX_VERTICAL_LEAP"]
display(pd.concat(combine.values()).query("PLAYER_ID in [1628983, 1642856]")[combine_cols])

DraftHistory: (8434, 14) 1947 → 2026


Combine 2018-19 (69, 47)


Combine 2025-26 (79, 47)


,PLAYER_ID,PLAYER_NAME,POSITION,HEIGHT_WO_SHOES,WEIGHT,WINGSPAN,STANDING_REACH,MAX_VERTICAL_LEAP
24,1628983,Shai Gilgeous-Alexander,PG,76.50,180,83.50,104.0,NaN
13,1642856,Egor Demin,PG-SG,80.25,199.20,82.25,105.5,32.0


### Other `LeagueDashPlayerStats` measure types

Which of these add information beyond Base + Advanced?

In [9]:
extra = {m: fetch_league_players(SEASON, m) for m in ["Usage", "Scoring", "Defense"]}
for m, d in extra.items():
    print(m, d.shape, [c for c in d.columns if not c.endswith("_RANK")][11:], "\n")

Usage (530, 53) ['USG_PCT', 'PCT_FGM', 'PCT_FGA', 'PCT_FG3M', 'PCT_FG3A', 'PCT_FTM', 'PCT_FTA', 'PCT_OREB', 'PCT_DREB', 'PCT_REB', 'PCT_AST', 'PCT_TOV', 'PCT_STL', 'PCT_BLK', 'PCT_BLKA', 'PCT_PF', 'PCT_PFD', 'PCT_PTS', 'TEAM_COUNT'] 

Scoring (530, 53) ['PCT_FGA_2PT', 'PCT_FGA_3PT', 'PCT_PTS_2PT', 'PCT_PTS_2PT_MR', 'PCT_PTS_3PT', 'PCT_PTS_FB', 'PCT_PTS_FT', 'PCT_PTS_OFF_TOV', 'PCT_PTS_PAINT', 'PCT_AST_2PM', 'PCT_UAST_2PM', 'PCT_AST_3PM', 'PCT_UAST_3PM', 'PCT_AST_FGM', 'PCT_UAST_FGM', 'FGM', 'FGA', 'FG_PCT', 'TEAM_COUNT'] 

Defense (530, 43) ['DEF_RATING', 'DREB', 'DREB_PCT', 'PCT_DREB', 'STL', 'PCT_STL', 'BLK', 'PCT_BLK', 'OPP_PTS_OFF_TOV', 'OPP_PTS_2ND_CHANCE', 'OPP_PTS_FB', 'OPP_PTS_PAINT', 'DEF_WS', 'DEF_WS_RAW'] 



### Step 1C findings

| Need | Source | Time-correct? | Notes |
|---|---|---|---|
| Height, weight (listed) | `LeagueDashPlayerBioStats` | ✅ per season | **Structural break in 2019-20**: most players got shorter (measured heights). |
| Age | Base / Bio `AGE` | ✅ | Birth date (for exact age) needs `CommonPlayerInfo`, one call per player. |
| Draft year/pick | `DraftHistory` (1 call) | ✅ static | Also in Bio. "Undrafted" is stored as a string, so we'll recode it. |
| Wingspan, reach, vertical | `DraftCombineStats` (1/yr) | ✅ pre-draft | Only covers combine attendees, so missingness is informative (many internationals skip it). |
| Position | `PlayerIndex` only | ❌ **current only** | Coarse (G/F/C) and leaky. |
| STL%, BLK%, TOV% | Usage (`PCT_STL`, …) = share of *team* totals; or compute `STL/POSS` ourselves | ✅ | The NBA's PCT_STL ≠ Basketball-Reference STL%; we'll define our own from totals. |
| Self-creation, shot mix | Scoring (`PCT_UAST_FGM`, `PCT_FGA_3PT`, `PCT_PTS_PAINT`) | ✅ | Unassisted-FG share is a strong candidate unicorn/breakout feature. |
| Rim frequency | `LeagueDashPlayerShotLocations` | (untested) | Step 1D, if needed. |
| BPM / VORP / WS / PER | Not on NBA.com | — | NBA's own all-in-ones: `PIE`; the Defense measure has `DEF_WS`. PER is computable from box score + league totals. |

### Season-specific listed position: `CommonTeamRoster`

Decision: use **listed position each season**. `PlayerIndex` is current-only, but team rosters are snapshots per season (Giannis: G-F in 2013-14 → F-G in 2016-17 → F in 2024-25). They also include `BIRTH_DATE` (exact age) and `EXP`. Cost: 30 calls per season.

Check: what share of the league-wide player-seasons have a roster position?

In [10]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster

rosters = []
for t in teams.get_teams():
    rosters.append(commonteamroster.CommonTeamRoster(team_id=t["id"], season=SEASON, timeout=60).get_data_frames()[0])
    time.sleep(0.8)
roster = pd.concat(rosters, ignore_index=True)

print("roster rows:", len(roster), "| unique players:", roster["PLAYER_ID"].nunique())
print(roster["POSITION"].value_counts().to_dict())

pos = df.merge(roster[["PLAYER_ID", "POSITION", "BIRTH_DATE", "EXP"]], on="PLAYER_ID", how="left", validate="one_to_one")
missing = pos[pos["POSITION"].isna()]
print(f"missing position: {len(missing)} of {len(pos)} players ({len(missing)/len(pos):.0%}), "
      f"{missing['MIN'].sum() / pos['MIN'].sum():.1%} of minutes")
missing.sort_values("MIN", ascending=False)[["PLAYER_NAME", "TEAM_ABBREVIATION", "TEAM_COUNT", "GP", "MIN"]].head(5)

roster rows: 497 | unique players: 497
{'G': 194, 'F': 159, 'C': 56, 'G-F': 33, 'F-C': 25, 'F-G': 16, 'C-F': 14}
missing position: 47 of 530 players (9%), 1.4% of minutes


,PLAYER_NAME,TEAM_ABBREVIATION,TEAM_COUNT,GP,MIN
79,Tyreke Evans,IND,1,69,1401.983333
31,Marcin Gortat,LAC,1,47,750.481667
279,Cameron Payne,CLE,2,40,712.225000
194,Isaiah Canaan,MIL,3,30,627.923333
212,Alex Abrines,OKC,1,31,588.273333


**Finding:** about 9% of player-seasons are missing a roster position but they account for about 1.4% of minutes. These are mostly players waived or released mid-season (e.g. Tyreke Evans, dismissed in 2018-19). Fill rule, to confirm in `02`: carry forward the player's most recent *earlier* roster position. Never use a later season.